# Demo: Strait of Hormuz closure scenario

End-to-end run of the impact mapper pipeline. This notebook is the backup
for the live UI demo on Saturday. It produces:

1. A `StructuredPolicy` parsed from plain English
2. A list of hypotheses from 3 specialists with cross-country diversity
3. Per-analog adversarial critiques
4. Regression plots for every edge
5. A rendered impact graph (networkx + matplotlib)
6. Review flags from the reviewer agent

Prereqs: `.env` with ANTHROPIC_API_KEY, FRED_API_KEY, HF_TOKEN.
Run `python -m src.loaders.core preload` once before this notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.orchestrator import run_impact_analysis
from src.agents._client import new_run_id

In [ ]:
SCENARIO = (
    'What happens to global markets if the Strait of Hormuz is closed for '
    '7 or more days with greater than 50 percent probability over the next 30 days? '
    'Roughly 20 percent of global oil trade passes through the strait.'
)
print(SCENARIO)

In [ ]:
# This runs the full pipeline. Takes ~2.5 to 3 min with 3 specialists.
impact = run_impact_analysis(
    SCENARIO,
    specialists_to_run=['monetary', 'supply_chain', 'international'],
)
run_id = impact.generation_metadata['run_id']
print(f'run_id: {run_id}')
print(f'edges: {len(impact.edges)} ({sum(1 for e in impact.edges if e.is_first_link)} first-link)')
print(f'review flags: {len(impact.review_flags)}')

## Parsed policy

In [ ]:
from pprint import pprint
pprint(impact.policy.model_dump())

## Hypotheses by specialist (with diversity check)

In [ ]:
import json
hyps = json.loads((ROOT / 'data' / 'runs' / run_id / 'hypotheses_enriched.json').read_text())
by_spec = {}
for h in hyps:
    by_spec.setdefault(h['proposed_by'], []).append(h)

for spec, hs in by_spec.items():
    print(f'=== {spec} ({len(hs)} hypotheses) ===')
    for h in hs:
        rat = h['economic_rationale'][:180]
        print(f'  {h["channel_id"]:<32}  {h["shock_variable"]:>18} -> {h["response_variable"]:<22}')
        print(f'    {rat}...')
    print()

## Sample adversarial critiques on historical analogs

Exactly the thing we don't want to miss: each analog gets scrutinized for
regime differences and concurrent shocks.

In [ ]:
count = 0
for h in hyps:
    for ep in h['historical_episodes']:
        if ep.get('adversarial_critique') and count < 6:
            print(f'[{h["proposed_by"]}] {ep["name"]} ({ep["date"]})')
            print(f'  -> {ep["adversarial_critique"]}')
            print()
            count += 1

## Edges with regression output

In [ ]:
print(f'{"first":<5} {"wave":<4} {"source":<28} -> {"target":<22} {"beta":>10} {"conf":>6}')
print('-' * 90)
for e in sorted(impact.edges, key=lambda e: (e.wave, -abs(e.elasticity.point or 0))):
    tag = 'FIRST' if e.is_first_link else ''
    print(f'{tag:<5} w{e.wave:<3} {e.source_node[:28]:<28} -> {e.target_node[:22]:<22} '
          f'{e.elasticity.point:>+10.4f} {e.confidence.overall:>6.2f}')

## Impact graph (networkx render)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

G = nx.DiGraph()
for n in impact.nodes:
    G.add_node(n.node_id, wave=n.wave, label=n.label)

# Add a virtual scenario node for first links
scenario_id = '__scenario__'
G.add_node(scenario_id, wave=0, label=impact.policy.subject)

for e in impact.edges:
    src = scenario_id if e.is_first_link else e.source_node
    G.add_edge(src, e.target_node, beta=e.elasticity.point,
               conf=e.confidence.overall, first=e.is_first_link)

# Wave-based layout: x = wave, y = spread
by_wave = {}
for nid, d in G.nodes(data=True):
    by_wave.setdefault(d.get('wave', 1), []).append(nid)
pos = {}
for wave, nids in by_wave.items():
    for i, nid in enumerate(nids):
        pos[nid] = (wave * 3, (i - len(nids)/2) * 1.2)

fig, ax = plt.subplots(figsize=(14, 8))
edge_colors = ['#4fb47a' if G[u][v]['beta'] > 0 else '#d35d5d' if G[u][v]['beta'] < 0 else '#888'
               for u, v in G.edges()]
edge_widths = [1 + 3 * min(abs(G[u][v]['beta']), 2) ** 0.5 for u, v in G.edges()]
edge_style = ['dashed' if G[u][v]['first'] else 'solid' for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_color='#161b20', edgecolors='#c879e3',
                       node_size=[500 if nid == scenario_id else 300 for nid in G.nodes()], ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       style=edge_style, alpha=0.7, arrows=True, arrowsize=14, ax=ax)
labels = {nid: G.nodes[nid].get('label', nid)[:20] for nid in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_color='#d8dde4', ax=ax)

ax.set_facecolor('#0b0d10')
fig.patch.set_facecolor('#0b0d10')
ax.axis('off')
ax.set_title(f'Impact map: {impact.policy.subject}', color='#d4a64a', fontsize=14)
plt.tight_layout()
plt.show()

## Review flags (what the reviewer caught)

In [ ]:
for f in impact.review_flags:
    print(f'[{f.severity:<7}] {f.category:<20} {f.target[:45]:<45}  {f.message[:140]}')

## Show one regression plot inline

In [ ]:
from IPython.display import Image, display

# Find the first edge that actually has a plot
for e in impact.edges:
    for m in e.method_estimates:
        if m.plot_path:
            print(f'{e.source_node} -> {e.target_node}  method={m.method}')
            print(f'  coef={m.coefficient}  SE={m.standard_error}  n={m.sample_size}  R^2={m.r_squared}')
            display(Image(filename=str(ROOT / m.plot_path)))
            break
    else:
        continue
    break

## Historical analogs at the scenario level

In [ ]:
for a in impact.historical_analogs:
    r30 = 'n/a' if a.get('response_30d_pct') is None else f'{a["response_30d_pct"]:+.2f}%'
    r90 = 'n/a' if a.get('response_90d_pct') is None else f'{a["response_90d_pct"]:+.2f}%'
    print(f'sim={a["similarity"]:.2f}  {a["event_date"]}  {a["event_name"][:45]:<45}  30d={r30}  90d={r90}')

## Data availability report

In [ ]:
impact.data_availability_report